# Milestone 1 Lab, notebook 5 of 6: The agent answers the question

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| 3. Ask the LLM with all the data | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| **5. The agent answers the question** (you are here) | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **about 8**. With the default model, the whole lab costs a few US cents.

## What this notebook is for

You now know what a tool and an agent are. Here we build the real analyst: an agent with six tools, answering the leader's question.

Then the comparison this whole lab has been building towards: the answer key, the single LLM call, and the agent, side by side.

In [ ]:
import os

import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    MAIN_QUESTION,
    OPENROUTER_BASE_URL,
    PREVIOUS_MONTH,
    AnalystAnswer,
    ToolInputError,
    check_figures,
    compare_profit,
    compare_with_key,
    connect,
    get_quality_flags,
    load_result,
    make_agent_tools,
    query_profit,
    save_result,
    show_answer,
    show_doc,
)
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI

client, MODEL = connect()
answer_key = load_result("answer_key", required_from="notebook 1")

## Step 1: The design

This is where the second design document comes in. The **HLD** (high-level design) names the parts of the system and what each one is responsible for.

In [ ]:
show_doc("HLD.md", "2. Components and data flow")

In one line, the flow is:

`question` → **analyst agent** ⇄ **tools** → `tables and metric definitions`

The most important part of the design is the split of work between the model and the code.

In [ ]:
show_doc("HLD.md", "4. Where AI is used and where it is not")

## Step 2: The six tools

In notebook 4 you built one tool. The analyst needs six. Here is what each is for.

In [ ]:
show_doc("HLD.md", "5. Interfaces")

Each one is a plain pandas function, built the same way as your `get_category_profit`. They are in `lab_helpers.py`, and their exact rules are in section 4 of `docs/LLD.md`. You don't need to read the code now. Let's just try them.

The top three products last month:

In [ ]:
pd.DataFrame(query_profit(LAST_MONTH, "product", top_n=3)["rows"])[["rank", "entity_id", "entity_name", "contribution_profit"]]

The change by category. Look at Camp: `change` is empty. The tool refuses to compare a partial total with a full one. The trap from notebook 1 is closed **by code**, so the model cannot fall into it.

In [ ]:
pd.DataFrame(compare_profit(LAST_MONTH, PREVIOUS_MONTH, "category")["rows"])[["entity_id", "current", "previous", "change", "is_complete"]]

The data problems for last month. The tool also returns the confidence level, worked out by the rule from notebook 1. The model doesn't judge confidence. It is told.

In [ ]:
quality = get_quality_flags(LAST_MONTH)

print("Confidence:", quality["confidence_level"])
for reason in quality["confidence_reasons"]:
    print("-", reason)

And if a tool is asked for something that doesn't exist, it says so and lists what is valid. The agent can read that message and correct itself.

In [ ]:
try:
    query_profit("2026-12", "product")
except ToolInputError as error:
    print(error)

## Step 3: Hand the tools to an agent

`make_agent_tools` wraps the six functions as LangChain tools, exactly as `tool(...)` did for one function in notebook 4.

It also gives us a `tool_log`: an empty list for now. Every time the agent calls a tool, the call and its result are written to this list. That log is how we will show where each number came from.

In [ ]:
agent_tools, tool_log = make_agent_tools()

for agent_tool in agent_tools:
    print(f"{agent_tool.name}: {agent_tool.description.splitlines()[0]}")

## Step 4: The agent's instructions

Compare this with the prompt in notebook 3. That one was over 800 words, mostly data. This one has **no data at all**. It has rules about using the tools.

In [ ]:
AGENT_SYSTEM_PROMPT = """You are the profit analyst for Kestrel Ridge Outfitters, a fictional online retailer of outdoor gear. You answer questions about profit for retail leaders.

Rules:
1. Every number you give must come from a tool result. Never add, subtract, rank, or round numbers yourself. If you need a total, a ranking, or a change, call the tool that returns it.
2. If a question says "last month", call get_reporting_period first to find out which month that is.
3. Before you give any figure, call get_quality_flags for the month you report on. Set confidence.level to the confidence_level the tool returned, and list every returned flag in caveats with its flag_id.
4. Put each number in figures exactly as the tool returned it, using the entity_id from the tool result. For a change between months use kind "change" and the later month. A value the tool returned as null is unavailable: report it as null and explain why.
5. In definitions_used, give the metric name and metric_version for every metric in your figures.
6. In sources, list the tables the tools read.
7. Decline, with status "declined" and a decline_reason, if a question asks for a forecast or for anything the tools cannot provide.
"""

## Step 5: Build the agent

This is the same `create_agent` as in notebook 4, with one new argument.

`response_format=ToolStrategy(AnalystAnswer)` tells the agent how to finish: by filling in the `AnalystAnswer` form from notebook 3. So we get the same shape of answer from the agent as from the single call, and we can compare them fairly.

In [ ]:
llm = ChatOpenAI(model=MODEL, base_url=OPENROUTER_BASE_URL, api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0, timeout=120, max_retries=5)

agent = create_agent(model=llm, tools=agent_tools, system_prompt=AGENT_SYSTEM_PROMPT, response_format=ToolStrategy(AnalystAnswer))

## Step 6: Ask the question

`invoke` runs the loop: the model asks for tools, the tools run, and it repeats until the model fills in its answer. This takes several model requests, so it may take a minute.

In [ ]:
print(MAIN_QUESTION)

tool_log.clear()  # start with an empty log, in case you run this cell more than once

result = agent.invoke({"messages": [{"role": "user", "content": MAIN_QUESTION}]})
agent_answer = result["structured_response"]

print(f"The agent made {len(tool_log)} tool calls and gave {len(agent_answer.figures)} figures")

> **If that cell failed, or shows 0 tool calls or 0 figures**, run it again. A weak model may skip the tools or send back an empty answer, and free models often do. Each run costs several model requests.

Before the answer itself, look at what the agent **did**. This is the tool log.

In [ ]:
pd.DataFrame([{"tool": call["name"], "asked for": call["arguments"]} for call in tool_log])

Read down that list. The model worked out a plan by itself: find out which month is "last month", get the rankings, get the changes, check for data problems. We never wrote that plan. We only supplied the tools and the rules.

Section 7 of `docs/LLD.md` has a diagram of the calls we expected. Open it and compare.

Now the answer.

In [ ]:
show_answer(agent_answer)

## Step 7: The comparison

Here it is: the answer key from notebook 1, the single LLM call from notebook 3, and the agent, line by line.

In [ ]:
saved_llm_answer = load_result("llm_answer")
llm_answer = AnalystAnswer(**saved_llm_answer) if saved_llm_answer else None

compare_with_key(answer_key, {"LLM": llm_answer, "Agent": agent_answer})

And every number the agent gave, checked against the truth.

In [ ]:
check_figures(agent_answer)

There is one more thing the agent can do that the single call never could. Pick any number in its answer, and we can show exactly where it came from.

In [ ]:
figure = agent_answer.figures[0]
print(f"The answer says: {figure.label} = {figure.value}")
print(f"It says this came from the tool: {figure.source_tool}\n")

for call in tool_log:
    if call["name"] == figure.source_tool:
        print("That tool was called with:", call["arguments"])
        print("It read these tables:", [source["table"] for source in call["result"]["sources"]])
        break

## Why the agent is the better design

Look at the comparison table again. With a capable model, **both** approaches probably got every number right. So the table of right and wrong answers is not the reason to prefer the agent. On 24 rows, a good model can do the arithmetic.

The reason is what each design lets you **rely on**. Both approaches used the same model. The difference is what we asked the model to do.

| | Single LLM call (notebook 3) | Agent with tools (this notebook) |
|---|---|---|
| **Who does the arithmetic?** | The model, by reading text | Code, in the tools. It gives the same result every time |
| **Where is the data?** | Pasted into the prompt. 24 rows fit; millions won't | Behind the tools. The prompt has no data in it |
| **Can we trace a number?** | No. There is nothing to point to | Yes. The tool log shows the call and the tables it read |
| **Who applies the business rules?** | The model, if it remembers (the Camp trap, the confidence rule) | The tools. Camp has no change; confidence is worked out by rule |
| **Can we check an answer with no answer key?** | No | Yes. Every figure can be matched to the tool log (notebook 6) |
| **What is the model's job?** | Everything | Understand the question, choose the tools, explain the result |

Notice also the kind of mistake each design can make. The single call can get a sum wrong and nothing will tell you. An agent that follows its rules can leave something **out**, but it has no way to get a sum **wrong**, because it never does the sum.

That is the lesson of this milestone: let the model do what it is good at, and keep every number in code.

Save the agent's answer and its tool log. The last notebook uses them.

In [ ]:
save_result("agent_answer", agent_answer.model_dump())
save_result("agent_tool_log", tool_log)

**Next: notebook 6**, where we check an answer with code alone, and see the production version of everything you have built.